In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import preprocessing as pp, feature_engineering as fe, pipeline as P
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
def show(name, w=800): display(Image(filename=str(FIG / name), width=w))

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 08 Baseline models
Logistic Regression, Decision Tree, Random Forest, XGBoost, plus a majority-class dummy. 80/20 stratified split; 5-fold CV grid search on the 80% (macro-F1). Set `RETRAIN=True` to re-run (~8 min).

In [2]:
RETRAIN = False
if RETRAIN:
    m = pd.read_csv(PROC/'irf_facilities_merged.csv', dtype={CCN:str}, parse_dates=['Certification Date'])
    df = pd.read_csv(PROC/'facility_quality_index.csv', dtype={CCN:str}); df['performance_category']=pd.Categorical(df['performance_category'], CLASS_ORDER)
    X, XB, y = P._feature_sets(m, df); A = P.run_experiment(X, y, 'A_characteristics')
pd.read_csv(TAB/'tuning_A_characteristics.csv')

,model,cv_macro_f1,cv_std,best_params
0,Logistic Regression,0.398041,0.021145,{'C': 1}
1,Decision Tree,0.379248,0.037831,"{'max_depth': 6, 'min_samples_leaf': 5}"
2,Random Forest,0.407778,0.034082,"{'max_depth': 8, 'max_features': 0.5, 'min_sam..."
3,XGBoost,0.419417,0.023306,"{'learning_rate': 0.03, 'max_depth': 4, 'n_est..."


In [3]:
t = pd.read_csv(TAB/'model_comparison_A_characteristics.csv'); t[t.Model.isin(['Majority-class baseline','Logistic Regression','Decision Tree','Random Forest','XGBoost'])].round(3)

,Model,Accuracy,Balanced Acc.,Precision (macro),Recall (macro),Specificity (macro),F1 (macro),F1 (weighted),ROC-AUC (OvR macro),PR-AUC (macro),Brier (multiclass),Low<->High errors (%),F1 (macro) 95% CI,CV macro-F1 (train)
0,Logistic Regression,0.387,0.387,0.384,0.387,0.693,0.384,0.384,0.580,0.408,0.655,17.778,"[0.321, 0.444]",0.398
1,Decision Tree,0.382,0.382,0.385,0.382,0.691,0.383,0.383,0.544,0.376,0.729,12.889,"[0.320, 0.445]",0.379
2,Random Forest,0.391,0.391,0.372,0.391,0.696,0.375,0.375,0.591,0.427,0.645,18.222,"[0.312, 0.433]",0.408
3,XGBoost,0.396,0.396,0.391,0.396,0.698,0.393,0.393,0.568,0.404,0.682,16.000,"[0.333, 0.450]",0.419
4,Majority-class baseline,0.333,0.333,0.111,0.333,0.667,0.167,0.167,0.500,0.333,0.667,33.333,"[0.144, 0.189]",NaN
